# LA Studio Qwen3 VoiceDesign GPU Worker

Choose **Runtime > Change runtime type > GPU**, then **Run all**. This starts a temporary direct Colab worker for Qwen3-TTS 1.7B VoiceDesign. It never reads, stores, or calls an API Gateway key. The final cell prints an HTTPS URL and temporary bearer token for LA Studio's **Colab GPU VoiceDesign** panel.

In [ ]:
import subprocess, sys

def run(*args):
    print('+', ' '.join(args))
    subprocess.run(args, check=True)

run('nvidia-smi')
run(sys.executable, '-m', 'pip', 'install', '--quiet', 'qwen-tts', 'soundfile>=0.12.1', 'fastapi>=0.115.0', 'uvicorn[standard]>=0.34.0')


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_voice_design_worker.py')
WORKER.write_text(r'''
import io
import os
import threading

import soundfile as sf
import torch
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime before starting this worker.')

TOKEN = os.environ['LA_STUDIO_COLAB_TOKEN']
MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign'
LANGUAGES = {'auto': 'Auto', 'zh': 'Chinese', 'en': 'English', 'ja': 'Japanese', 'ko': 'Korean', 'de': 'German', 'fr': 'French', 'ru': 'Russian', 'pt': 'Portuguese', 'es': 'Spanish', 'it': 'Italian'}
MODEL_LOCK = threading.Lock()
MAX_INPUT_CHARS = 4000
MAX_OUTPUT_SECONDS = 300
REQUEST_SLOTS = threading.BoundedSemaphore(1)

class VoiceDesignRequest(BaseModel):
    model: str = Field(min_length=1, max_length=120)
    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)
    voice_description: str = Field(min_length=1, max_length=2000)
    style: str = Field(default='', max_length=1000)
    language: str = 'en'
    temperature: float = Field(default=0.9, ge=0.1, le=2.0)
    seed: int = Field(default=-1, ge=-1, le=2147483647)
    response_format: str = 'wav'

def require_token(authorization: str | None) -> None:
    if authorization != 'Bearer ' + TOKEN:
        raise HTTPException(status_code=401, detail='invalid worker token')

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID, device_map='cuda:0', dtype=torch.float16, attn_implementation='sdpa'
)
app = FastAPI(title='LA Studio Colab VoiceDesign Worker', docs_url=None, redoc_url=None, openapi_url=None)

@app.get('/health')
@app.get('/v1/health')
def health(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'status': 'ready', 'ready': True, 'device': 'cuda', 'gpu': torch.cuda.get_device_name(0), 'api_version': '1.0'}

@app.get('/v1/capabilities')
def capabilities(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'contract_version': 1, 'capabilities': [{'id': 'voice-design', 'models': [{'id': 'qwen3-tts-1.7b-voicedesign', 'upstream_model': MODEL_ID, 'languages': sorted(LANGUAGES), 'formats': ['wav'], 'device': 'cuda'}]}]}

@app.post('/v1/audio/voice_designs')
def voice_design(request: VoiceDesignRequest, authorization: str | None = Header(default=None)):
    require_token(authorization)
    if request.model.strip().lower() not in {'qwen3-tts-1.7b-voicedesign', 'qwen3-tts-12hz-1.7b-voicedesign'}:
        raise HTTPException(status_code=422, detail='this worker supports qwen3-tts-1.7b-voicedesign only')
    language = LANGUAGES.get(request.language.strip().lower())
    if not language:
        raise HTTPException(status_code=422, detail='unsupported Qwen3 VoiceDesign language')
    instruction = request.voice_description.strip()
    if request.style.strip():
        instruction += '\nStyle: ' + request.style.strip()
    if not REQUEST_SLOTS.acquire(blocking=False):
        raise HTTPException(status_code=429, detail='the Colab Voice Design worker is busy; retry shortly')
    try:
        with MODEL_LOCK, torch.inference_mode():
            if request.seed >= 0:
                torch.manual_seed(request.seed)
                torch.cuda.manual_seed_all(request.seed)
            wavs, sample_rate = model.generate_voice_design(
                text=request.input.strip(), language=language, instruct=instruction,
                do_sample=True, temperature=request.temperature
            )
        if not wavs:
            raise HTTPException(status_code=500, detail='VoiceDesign returned no audio')
        if len(wavs[0]) > sample_rate * MAX_OUTPUT_SECONDS:
            raise HTTPException(status_code=413, detail='generated audio exceeds the five minute output limit')
        output = io.BytesIO()
        sf.write(output, wavs[0], sample_rate, format='WAV', subtype='PCM_16')
        return Response(content=output.getvalue(), media_type='audio/wav', headers={'Cache-Control': 'no-store'})
    finally:
        REQUEST_SLOTS.release()
''')
print('Worker source written:', WORKER)


In [ ]:
import os, re, secrets, subprocess, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env['LA_STUDIO_COLAB_TOKEN'] = TOKEN
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'la_studio_voice_design_worker:app', '--host', '127.0.0.1', '--port', '3922'], cwd='/content', env=env)
for _ in range(45):
    try:
        request = urllib.request.Request('http://127.0.0.1:3922/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=3) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError('LA Studio VoiceDesign worker did not become ready')
subprocess.run(['bash', '-lc', 'wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb'], check=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3922', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[^\s]+trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate(); tunnel.terminate()
    raise RuntimeError('Cloudflare tunnel URL was not found')
print('\nLA_STUDIO_COLAB_VOICE_DESIGN_URL=' + public_url)
print('LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN=' + TOKEN)
print('MODEL=qwen3-tts-1.7b-voicedesign  LANGUAGE=en')


In [ ]:
# ==============================================================================
# 📥 LƯU FILE TRỰC TIẾP VÀO THƯ MỤC DỰ ÁN TRÊN MÁY TÍNH (FILE SYSTEM ACCESS API)
# ==============================================================================
import base64
import glob
import json
import os
from IPython.display import HTML, display

# Thu thập tất cả các file kết quả vừa tạo
result_files = {}
for pattern in ['/content/*.wav', '/content/*.srt', '/content/*.json', '/content/*/*/*.wav', '/content/*/*/*.srt']:
    for f in glob.glob(pattern):
        name = os.path.basename(f)
        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:
            with open(f, 'rb') as fp:
                result_files[name] = base64.b64encode(fp.read()).decode('utf-8')

if not result_files:
    print("⚠️ Chưa có file kết quả mới để lưu.")
else:
    print(f"✅ Đã tìm thấy {len(result_files)} file kết quả: {', '.join(result_files.keys())}")
    print("👉 Bấm nút bên dưới và chọn thư mục 'LA-Studio/out/colab-live' để lưu thẳng vào máy:")
    
    files_json = json.dumps(result_files)
    html_code = f"""
    <button id="saveBtn" style="background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;">
        📁 Chọn Thư Mục & Lưu File Trực Tiếp Vào Máy
    </button>
    <div id="statusLog" style="margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;"></div>
    <script>
    document.getElementById('saveBtn').onclick = async () => {{
        const log = document.getElementById('statusLog');
        try {{
            if (!window.showDirectoryPicker) {{
                log.innerText = 'Trình duyệt không hỗ trợ File System Access API. Đang dùng tải thông thường...';
                return;
            }}
            log.innerText = 'Đang mở hộp thoại chọn thư mục...';
            const dirHandle = await window.showDirectoryPicker();
            const files = {files_json};
            for (const [name, b64] of Object.entries(files)) {{
                log.innerText = 'Đang ghi file: ' + name + '...';
                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});
                const writable = await fileHandle.createWritable();
                const byteCharacters = atob(b64);
                const byteNumbers = new Array(byteCharacters.length);
                for (let i = 0; i < byteCharacters.length; i++) {{
                    byteNumbers[i] = byteCharacters.charCodeAt(i);
                }}
                const byteArray = new Uint8Array(byteNumbers);
                await writable.write(byteArray);
                await writable.close();
            }}
            log.innerText = '🎉 Đã lưu thành công toàn bộ file vào thư mục bạn chọn!';
        }} catch (err) {{
            if (err.name !== 'AbortError') {{
                log.innerText = 'Lỗi: ' + err.message;
            }} else {{
                log.innerText = 'Đã hủy chọn thư mục.';
            }}
        }}
    }};
    </script>
    """
    display(HTML(html_code))
